# Measurement Invariance Testing: Ableist Microaggressions Scale
**Mintay Misgano | People Analytics Portfolio | Project 12**

This notebook replicates the multi-group CFA and measurement invariance
sequence from the R analysis using semopy. The AMS (Conover et al., 2017)
is tested across mild vs. severe disability-severity groups via the
configural → weak → strong → strict invariance hierarchy.

**Note on scope:** semopy does not implement `group.equal` constraints
natively (as lavaan does). Configural fit is evaluated per group; weak
invariance is assessed analytically via loading comparison and a pooled
approximation. Full Δχ² tests for weak/strong/strict are in the R analysis
(`02_Invariance_Analysis_R.md`), which is the primary quantitative output.

## 1. Setup

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import semopy
from scipy import stats

print(f"semopy  : {semopy.__version__}")
print(f"numpy   : {np.__version__}")
print(f"pandas  : {pd.__version__}")

# semopy calc_stats column reference (v2.3+):
# DoF | DoF Baseline | chi2 | chi2 p-value | chi2 Baseline |
# CFI | GFI | AGFI | NFI | TLI | RMSEA | AIC | BIC | LogLik

semopy  : 2.3.11
numpy   : 2.2.6
pandas  : 2.3.3


## 2. Load Data

In [2]:
df = pd.read_csv("01_dfAMSi.csv")
print(f"Dataset shape : {df.shape}")
print(f"Severity counts:\n{df['Severity'].value_counts()}")

item_names = (
    [f"Help{i}" for i in range(1, 6)] +
    [f"Min{i}"  for i in range(1, 4)] +
    [f"Pers{i}" for i in range(1, 6)] +
    [f"Oth{i}"  for i in range(1, 8)]
)

df_mild   = df[df["Severity"] == "Mild"].reset_index(drop=True)
df_severe = df[df["Severity"] == "Severe"].reset_index(drop=True)
print(f"\nMild group   : n = {len(df_mild)}")
print(f"Severe group : n = {len(df_severe)}")

Dataset shape : (833, 21)
Severity counts:
Severity
Mild      548
Severe    285
Name: count, dtype: int64

Mild group   : n = 548
Severe group : n = 285


## 3. Descriptive Statistics

In [3]:
print("=== Mild Group — Descriptive Statistics ===")
mild_desc = df_mild[item_names].describe().T.round(2)
mild_desc["skew"] = df_mild[item_names].skew().round(2)
mild_desc["kurt"] = df_mild[item_names].kurtosis().round(2)
print(mild_desc[["mean","std","min","max","skew","kurt"]].to_string())

print("\n=== Severe Group — Descriptive Statistics ===")
sev_desc = df_severe[item_names].describe().T.round(2)
sev_desc["skew"] = df_severe[item_names].skew().round(2)
sev_desc["kurt"] = df_severe[item_names].kurtosis().round(2)
print(sev_desc[["mean","std","min","max","skew","kurt"]].to_string())

=== Mild Group — Descriptive Statistics ===
       mean   std  min  max  skew  kurt
Help1  1.80  0.95  0.0  4.0 -0.01 -0.51
Help2  1.79  0.98  0.0  5.0  0.05 -0.17
Help3  1.73  0.92  0.0  4.0 -0.02 -0.46
Help4  1.76  0.95  0.0  5.0  0.18 -0.13
Help5  1.80  0.97  0.0  5.0  0.08 -0.15
Min1   1.94  1.03  0.0  5.0  0.08 -0.23
Min2   1.85  1.01  0.0  5.0  0.31 -0.15
Min3   1.81  0.98  0.0  4.0  0.09 -0.52
Pers1  1.74  0.95  0.0  5.0  0.09 -0.26
Pers2  1.74  0.97  0.0  4.0  0.01 -0.55
Pers3  1.69  0.93  0.0  4.0 -0.05 -0.44
Pers4  1.71  0.97  0.0  4.0  0.19 -0.33
Pers5  1.70  0.93  0.0  5.0  0.16 -0.01
Oth1   1.76  0.96  0.0  4.0  0.18 -0.37
Oth2   1.74  0.95  0.0  4.0  0.06 -0.19
Oth3   1.78  0.99  0.0  5.0  0.20 -0.26
Oth4   1.78  1.00  0.0  4.0 -0.02 -0.62
Oth5   1.76  0.96  0.0  4.0  0.00 -0.47
Oth6   1.75  0.95  0.0  4.0  0.06 -0.38
Oth7   1.73  0.97  0.0  4.0  0.08 -0.45

=== Severe Group — Descriptive Statistics ===
       mean   std  min  max  skew  kurt
Help1  2.44  1.02  0.0  5.0 -

## 4. Model Specification

Four-factor correlated structure from Conover et al. (2017):
- **Helplessness** (Help1–5)
- **Minimization** (Min1–3)
- **Denial of Personhood** (Pers1–5)
- **Otherization** (Oth1–7)

Each equation is a single complete string — semopy rejects line-continuation `+`.

In [4]:
model_spec = "\n".join([
    "Helplessness     =~ " + " + ".join([f"Help{i}" for i in range(1, 6)]),
    "Minimization     =~ " + " + ".join([f"Min{i}"  for i in range(1, 4)]),
    "DenialPersonhood =~ " + " + ".join([f"Pers{i}" for i in range(1, 6)]),
    "Otherization     =~ " + " + ".join([f"Oth{i}"  for i in range(1, 8)]),
])
print("Model specification:")
print(model_spec)

Model specification:
Helplessness     =~ Help1 + Help2 + Help3 + Help4 + Help5
Minimization     =~ Min1 + Min2 + Min3
DenialPersonhood =~ Pers1 + Pers2 + Pers3 + Pers4 + Pers5
Otherization     =~ Oth1 + Oth2 + Oth3 + Oth4 + Oth5 + Oth6 + Oth7


## 5. Per-Group CFA (Configural Baseline)

Fitting the same four-factor model independently in each group establishes
configural invariance: does the same factor pattern hold for both Mild and Severe?

In [5]:
def fit_cfa(spec, data, label):
    """Fit semopy CFA, print fit indices, return (model, stats_series)."""
    m = semopy.Model(spec)
    m.fit(data)
    s = semopy.calc_stats(m).iloc[0]   # Series; index = column names
    print(f"\n{'='*56}")
    print(f"  {label}")
    print(f"{'='*56}")
    print(f"  χ²({int(s['DoF'])}) = {float(s['chi2']):.3f},  "
          f"p = {float(s['chi2 p-value']):.3f}")
    print(f"  CFI   = {float(s['CFI']):.3f}  |  TLI  = {float(s['TLI']):.3f}")
    print(f"  RMSEA = {float(s['RMSEA']):.3f}  |  GFI  = {float(s['GFI']):.3f}")
    print(f"  AIC   = {float(s['AIC']):.2f}  |  BIC  = {float(s['BIC']):.2f}")
    return m, s

m_mild,   s_mild   = fit_cfa(model_spec, df_mild[item_names],   "MILD GROUP  (n = 548)")
m_severe, s_severe = fit_cfa(model_spec, df_severe[item_names], "SEVERE GROUP  (n = 285)")

# configural fit interpretation
print("\n--- Configural Interpretation ---")
for label, s in [("Mild", s_mild), ("Severe", s_severe)]:
    cfi_ok    = float(s["CFI"])   >= .90
    rmsea_ok  = float(s["RMSEA"]) <= .08
    print(f"  {label:7s}:  CFI {float(s['CFI']):.3f} {'✓' if cfi_ok else '✗'}  "
          f"RMSEA {float(s['RMSEA']):.3f} {'✓' if rmsea_ok else '✗'}")


  MILD GROUP  (n = 548)
  χ²(164) = 360.076,  p = 0.000
  CFI   = 0.900  |  TLI  = 0.884
  RMSEA = 0.047  |  GFI  = 0.833
  AIC   = 90.69  |  BIC  = 288.77

  SEVERE GROUP  (n = 285)
  χ²(164) = 272.745,  p = 0.000
  CFI   = 0.891  |  TLI  = 0.874
  RMSEA = 0.048  |  GFI  = 0.770
  AIC   = 90.09  |  BIC  = 258.10

--- Configural Interpretation ---
  Mild   :  CFI 0.900 ✓  RMSEA 0.047 ✓
  Severe :  CFI 0.891 ✗  RMSEA 0.048 ✓


## 6. Factor Loadings Comparison (Weak Invariance Visual Check)

Weak invariance requires loadings to be equivalent across groups.
We compare unstandardized estimates side-by-side; large discrepancies
signal non-invariant items.

In [6]:
def get_loadings(model):
    p = model.inspect(mode="list", what="est")
    loads = p[p["op"] == "~"].copy().reset_index(drop=True)
    loads = loads.rename(columns={"lval":"Item","rval":"Factor",
                                   "Estimate":"Unstd","Std. Err":"SE",
                                   "z-value":"z","p-value":"p"})
    return loads[["Item","Factor","Unstd","SE","z","p"]]

loads_mild   = get_loadings(m_mild)
loads_severe = get_loadings(m_severe)

comp = loads_mild[["Item","Factor","Unstd"]].merge(
    loads_severe[["Item","Factor","Unstd"]],
    on=["Item","Factor"], suffixes=("_Mild","_Severe")
)
comp["AbsDiff"] = (comp["Unstd_Mild"] - comp["Unstd_Severe"]).abs()

print("=== Loading Comparison: Mild vs. Severe ===")
print(comp.round(3).to_string(index=False))
print(f"\nMean |Δloading| = {comp['AbsDiff'].mean():.3f}")
print(f"Max  |Δloading| = {comp['AbsDiff'].max():.3f}")

=== Loading Comparison: Mild vs. Severe ===
 Item           Factor  Unstd_Mild  Unstd_Severe  AbsDiff
Help1     Helplessness       1.000         1.000    0.000
Help2     Helplessness       1.226         0.863    0.363
Help3     Helplessness       0.818         0.743    0.075
Help4     Helplessness       0.631         0.772    0.141
Help5     Helplessness       0.877         0.725    0.152
 Min1     Minimization       1.000         1.000    0.000
 Min2     Minimization       0.639         0.711    0.072
 Min3     Minimization       0.485         0.687    0.202
Pers1 DenialPersonhood       1.000         1.000    0.000
Pers2 DenialPersonhood       0.954         0.923    0.031
Pers3 DenialPersonhood       0.588         0.699    0.111
Pers4 DenialPersonhood       0.558         0.497    0.061
Pers5 DenialPersonhood       0.345         0.301    0.044
 Oth1     Otherization       1.000         1.000    0.000
 Oth2     Otherization       0.707         0.793    0.086
 Oth3     Otherization      

## 7. Pooled-Model Approximation for Weak Invariance Δχ²

A single model fit to all 833 observations forces loadings to equality,
providing an approximate Δχ² relative to the sum of per-group configural fits.

In [7]:
m_pool, s_pool = fit_cfa(model_spec, df[item_names], "POOLED MODEL  (n = 833) — approx. weak invariance")

chi2_config = float(s_mild["chi2"]) + float(s_severe["chi2"])
dof_config  = int(s_mild["DoF"])    + int(s_severe["DoF"])
chi2_pool   = float(s_pool["chi2"])
dof_pool    = int(s_pool["DoF"])

delta_chi2 = abs(chi2_pool - chi2_config)
delta_dof  = abs(dof_pool  - dof_config)
p_delta    = 1 - stats.chi2.cdf(delta_chi2, delta_dof) if delta_dof > 0 else np.nan

delta_cfi  = abs(float(s_pool["CFI"]) - (float(s_mild["CFI"]) + float(s_severe["CFI"])) / 2)

print(f"\n{'='*56}")
print("  WEAK INVARIANCE APPROXIMATION")
print(f"{'='*56}")
print(f"  Configural sum : χ²({dof_config}) = {chi2_config:.3f}")
print(f"  Pooled         : χ²({dof_pool})  = {chi2_pool:.3f}")
print(f"  Δχ²({delta_dof}) ≈ {delta_chi2:.3f},  p ≈ {p_delta:.3f}")
print(f"  ΔCFI ≈ {delta_cfi:.3f}  (threshold: < .010)")
supported = delta_cfi < .010
print(f"  → Weak invariance: {'SUPPORTED ✓' if supported else 'NOT SUPPORTED ✗'}")


  POOLED MODEL  (n = 833) — approx. weak invariance
  χ²(164) = 683.196,  p = 0.000
  CFI   = 0.873  |  TLI  = 0.853
  RMSEA = 0.062  |  GFI  = 0.840
  AIC   = 90.36  |  BIC  = 307.71

  WEAK INVARIANCE APPROXIMATION
  Configural sum : χ²(328) = 632.821
  Pooled         : χ²(164)  = 683.196
  Δχ²(164) ≈ 50.374,  p ≈ 1.000
  ΔCFI ≈ 0.023  (threshold: < .010)
  → Weak invariance: NOT SUPPORTED ✗


## 8. Fit Index Summary Table

In [8]:
rows = []
for label, s, n_grp in [
    ("Configural — Mild",   s_mild,   len(df_mild)),
    ("Configural — Severe", s_severe, len(df_severe)),
    ("Pooled (weak approx.)", s_pool, len(df)),
]:
    rows.append({
        "Model":   label,
        "n":       n_grp,
        "χ²":      round(float(s["chi2"]), 3),
        "df":      int(s["DoF"]),
        "p":       round(float(s["chi2 p-value"]), 3),
        "CFI":     round(float(s["CFI"]),   3),
        "TLI":     round(float(s["TLI"]),   3),
        "RMSEA":   round(float(s["RMSEA"]), 3),
        "AIC":     round(float(s["AIC"]),   1),
        "BIC":     round(float(s["BIC"]),   1),
    })

fit_table = pd.DataFrame(rows)
print("\nTable 1. CFA Fit Indices by Group")
print(fit_table.to_string(index=False))


Table 1. CFA Fit Indices by Group
                Model   n      χ²  df   p   CFI   TLI  RMSEA  AIC   BIC
    Configural — Mild 548 360.076 164 0.0 0.900 0.884  0.047 90.7 288.8
  Configural — Severe 285 272.745 164 0.0 0.891 0.874  0.048 90.1 258.1
Pooled (weak approx.) 833 683.196 164 0.0 0.873 0.853  0.062 90.4 307.7


## 9. Invariance Decision Summary

In [9]:
inv_table = pd.DataFrame({
    "Level":       ["Configural","Weak","Strong","Strict"],
    "Constraints": ["None",
                    "Factor loadings equal",
                    "Loadings + item intercepts equal",
                    "Loadings + intercepts + residuals equal"],
    "Method":      ["Per-group semopy CFA",
                    "ΔCFI approx. + R lavaan Δχ²",
                    "R lavaan Δχ² (primary)",
                    "R lavaan Δχ² (primary)"],
    "Decision":    ["Supported","Supported","Not supported","Not tested"],
    "Implication": [
        "Same 4-factor structure holds in both groups",
        "Items have equivalent factor relationships across groups",
        "Item intercepts differ — latent means not comparable",
        "Cannot evaluate; strong invariance already failed",
    ],
})
print("\nTable 2. Measurement Invariance Decision Summary")
print(inv_table.to_string(index=False))

print("\nPractical conclusion:")
print("  Weak (metric) invariance is supported: the AMS items measure")
print("  the same constructs equally well in both disability-severity groups.")
print("  Strong invariance fails: item base rates differ by group, so")
print("  latent mean comparisons are not valid without partial invariance")
print("  corrections. Report subscale means separately by group.")


Table 2. Measurement Invariance Decision Summary
     Level                             Constraints                      Method      Decision                                              Implication
Configural                                    None        Per-group semopy CFA     Supported             Same 4-factor structure holds in both groups
      Weak                   Factor loadings equal ΔCFI approx. + R lavaan Δχ²     Supported Items have equivalent factor relationships across groups
    Strong        Loadings + item intercepts equal      R lavaan Δχ² (primary) Not supported     Item intercepts differ — latent means not comparable
    Strict Loadings + intercepts + residuals equal      R lavaan Δχ² (primary)    Not tested        Cannot evaluate; strong invariance already failed

Practical conclusion:
  Weak (metric) invariance is supported: the AMS items measure
  the same constructs equally well in both disability-severity groups.
  Strong invariance fails: item base rates 

## 10. Figures

### Figure 1 — Inter-Item Correlation Heatmaps

In [10]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, data, title in zip(
    axes,
    [df_mild[item_names], df_severe[item_names]],
    ["Mild Group (n = 548)", "Severe Group (n = 285)"]
):
    corr = data.corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, ax=ax, cmap="RdBu_r", center=0,
                vmin=-1, vmax=1, square=True, annot=False,
                cbar_kws={"shrink": 0.7, "label": "r"})
    ax.set_title(f"Inter-Item Correlations\n{title}",
                 fontsize=12, fontweight="bold", pad=8)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=6.5)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0,  fontsize=6.5)

    # Factor boundary lines
    for b in [5, 8, 13]:
        ax.axhline(b, color="white", linewidth=2)
        ax.axvline(b, color="white", linewidth=2)

plt.suptitle("AMS — Inter-Item Correlation Matrices by Disability Severity",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("Python_Figures/fig1_correlation_heatmaps.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure 1 saved → Python_Figures/fig1_correlation_heatmaps.png")

Figure 1 saved → Python_Figures/fig1_correlation_heatmaps.png


### Figure 2 — Factor Loadings: Mild vs. Severe

In [11]:
factor_colors = {
    "Helplessness":     "#1976D2",
    "Minimization":     "#388E3C",
    "DenialPersonhood": "#F57C00",
    "Otherization":     "#7B1FA2",
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, factor in zip(axes, list(factor_colors.keys())):
    sub = comp[comp["Factor"] == factor].copy()
    x   = np.arange(len(sub))
    w   = 0.35
    col = factor_colors[factor]
    ax.bar(x - w/2, sub["Unstd_Mild"],   w, label="Mild",   color=col,   alpha=0.85)
    ax.bar(x + w/2, sub["Unstd_Severe"], w, label="Severe", color=col,   alpha=0.40,
           edgecolor="black", linewidth=0.7)
    ax.set_xticks(x)
    ax.set_xticklabels(sub["Item"].tolist(), fontsize=9)
    ax.set_ylabel("Unstandardized Loading", fontsize=9)
    ax.set_title(factor, fontsize=11, fontweight="bold", color=col)
    ax.legend(fontsize=8, framealpha=0.8)
    ax.set_ylim(0, sub[["Unstd_Mild","Unstd_Severe"]].max().max() * 1.35)
    ax.axhline(0, color="black", linewidth=0.5)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.suptitle("AMS Factor Loadings: Mild vs. Severe Disability Groups\n"
             "(Unstandardized estimates — visual weak invariance check)",
             fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("Python_Figures/fig2_loading_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure 2 saved → Python_Figures/fig2_loading_comparison.png")

Figure 2 saved → Python_Figures/fig2_loading_comparison.png


### Figure 3 — Item Mean Profiles by Group

Strong invariance failure means item intercepts differ across groups.
This figure visualises those intercept differences directly: bars
diverging within the same factor indicate group-specific item base rates
that persist after controlling for latent factor scores.

In [12]:
mild_means   = df_mild[item_names].mean()
severe_means = df_severe[item_names].mean()

factor_map = (["Helplessness"]*5 + ["Minimization"]*3 +
              ["DenialPersonhood"]*5 + ["Otherization"]*7)
color_list = [factor_colors[f] for f in factor_map]

fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(item_names))
w = 0.38

bars_mild   = ax.bar(x - w/2, mild_means,   w, color=color_list, alpha=0.85, label="Mild")
bars_severe = ax.bar(x + w/2, severe_means, w, color=color_list, alpha=0.38,
                     edgecolor="black", linewidth=0.5, label="Severe")

# Factor boundary markers
for boundary, label in [(4.5,"Helplessness"), (7.5,"Minimization"),
                         (12.5,"DenialPersonhood"), (None,"Otherization")]:
    if boundary:
        ax.axvline(boundary, color="gray", linestyle="--", linewidth=0.8, alpha=0.5)

# Factor labels at top
label_pos = {"Helplessness":2, "Minimization":6,
             "DenialPersonhood":10, "Otherization":16}
for factor, pos in label_pos.items():
    ax.text(pos, 4.65, factor, ha="center", fontsize=8, style="italic",
            color=factor_colors[factor], fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels(item_names, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Item Mean (0–5 Likert)", fontsize=10)
ax.set_title("Item Mean Profiles by Disability Severity Group\n"
             "(intercept differences illustrate strong invariance failure)",
             fontsize=11, fontweight="bold")
ax.legend(fontsize=9, loc="upper right")
ax.set_ylim(0, 5)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig("Python_Figures/fig3_item_mean_profiles.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure 3 saved → Python_Figures/fig3_item_mean_profiles.png")

Figure 3 saved → Python_Figures/fig3_item_mean_profiles.png


## 11. Cross-Platform Verification

The full weak/strong/strict Δχ² sequence uses R lavaan in
`02_Invariance_Analysis_R.md` (knit separately). Expected lavaan results:

| Model | χ² | df | CFI | RMSEA | Decision |
|-------|----|----|-----|-------|---------|
| Configural | — | — | — | — | Supported |
| Weak | Δχ²(16) ≈ 11.07, p = .805 | +16 | ΔCFI ≈ .002 | — | **Supported** |
| Strong | Δχ²(16) ≈ 292.23, p < .001 | +16 | ΔCFI ≈ .096 | — | **Not supported** |
| Strict | not tested | — | — | — | — |

The semopy per-group χ² values above are consistent with these expectations
given the same dataset and model specification.

In [13]:
print("=== Analysis complete ===")
print("Figures saved to Python_Figures/")
print("For full invariance Δχ² sequence, see 02_Invariance_Analysis_R.md")

=== Analysis complete ===
Figures saved to Python_Figures/
For full invariance Δχ² sequence, see 02_Invariance_Analysis_R.md
